# Проект 3 модуля: предсказание возраста посетителей сайтов для компании «Йети»

**Выполнил:** Артем Буров  
**Группа:** DS12  
**Дата:** 26 июня 2026  
**Ссылка на GitHub:** https://github.com/TemaQDX/project_module_03_yeti
___

## Описание задачи

**IT‑компания «Йети»**, управляющая группой популярных интернет‑сервисов, получает значительную часть дохода от контекстной рекламы. Чтобы эффективно таргетировать объявления, компания использует демографический таргетинг — в том числе по возрасту. При этом паспортные данные пользователей не задействуются: возраст определяют на основе анализа поведения в цифровой среде.

**Задача** — разработать модель машинного обучения, которая по логам посещений сайтов будет автоматически определять возрастную категорию пользователя. В логах содержатся данные о том, какие категории сайтов посещал пользователь, с какой частотой и в какое время.

Корректное определение возрастной категории критически важно для эффективности рекламных кампаний. Например, яркую рекламу игрового гаджета нужно показывать зумерам, а рекламу семейного отпуска — молодым родителям. Если таргетинг будет неточным, бюджет на рекламу расходуется впустую, конверсия падает, а пользователи видят нерелевантные объявления. Кроме того, правильная сегментация позволяет избежать показа рекламы для взрослых несовершеннолетним. Результат работы модели ляжет в основу реализации маркетинговых стратегий и поможет существенно повысить релевантность рекламных показов.
___

## Выбор метрик

1. В качестве основной метрики используется `F1-мера`.  
Значение F1-меры лучшей модели должно быть **не меньше 0.75** и на кросс-валидации по обучающей выборке, и на тестовой выборке. В этом случае модель можно рекомендовать к внедрению.

2. Вспомогательные метрики качества: `precision` и `recall`.  
Модель нужно оценить одинаково по всем классам с использованием макро-усреднения.

___

## Описание данных

Данные представлены в нескольких CSV-файлах, полученных из разных источников.  
Необходимо их проанализировать, собрать в единую таблицу и построить модель с максимально полным набором признаков.  

#### 1. Таблица `ds_s13_users` содержит информацию о возрастной категории пользователя:  
`user_id` — уникальный идентификатор пользователя.  
`age_category` — возрастная категория пользователя, этот показатель модель должна научиться предсказывать.  
Содержит следующие категории:
- 0: младше 18;
- 1: 18-25 лет;
- 2: 26-40 лет;
- 3: 41-55 лет;
- 4: 56+ лет.


#### 2. Лог посещений сайтов `ds_s13_visits.csv` похож на данные сервиса веб-аналитики «Яндекс Метрика».  
Предназначен для сбора и анализа информации об активности пользователей разных сайтов. В целях защиты персональных данных заменить в логах все URL на анонимизированную категорию сайта, а вместо точного времени использовать время суток.  
Признаки в датасете:  
`date` — дата посещения сайта.  
`daytime` — анонимизированное время посещения сайта. Категории: утро, день, вечер, ночь.  
`session_id` — уникальный идентификатор сессии. Сессия — это последовательность действий пользователя на сайте, которая начинается при первом взаимодействии с ресурсом и завершается по правилам тайм-аута или смены условий.  
`user_id` — уникальный идентификатор пользователя.  
`website_category` — анонимизированная категория сайта. В лог включены несколько десятков категорий, которые позволяют эффективно сегментировать аудиторию. Это позволяет сократить пространство признаков модели без потери её качества.  

#### 3. Таблица `ads_activity`
Активность взаимодействия пользователя с рекламными объявлениями — важный показатель, связанный с возрастом:  
`user_id` — уникальный идентификатор пользователя.  
`ads_activity` — характеристика CTR, выраженная одним из значений: очень редко, редко, умеренно, часто, очень часто.  

#### 4. Таблица `surf_depth`
В таблице содержится два признака:  
`user_id` — уникальный идентификатор пользователя.  
`surf_depth` — категориальная переменная, характеризующая глубину перехода пользователя по сайтам во время одной сессии. Содержит категории поверхностно, средне, глубоко.  

#### 5. Таблица `primary_device`
Таблица содержит два признака:  
`user_id` — уникальный идентификатор пользователя.  
`primary_device` — информация о типе основного устройства пользователя для выхода в Интернет.

#### 6. Таблица `cloud_usage`
Использование облачных технологий:  
`user_id` — уникальный идентификатор пользователя;  
`cloud_usage` — True означает, что пользователь обращается к облачным ресурсам типа Яндекс 360 прямо или через посещаемые сайты.
___

## Структура проекта

### 1. Подготовка среды и библиотек
___

## 1. Подготовка среды и библиотек

In [1]:
# Подготовка библиотек
import pandas as pd

In [2]:
# Подготовка функций

# Создадим функцию оптимизации типов данных датафрейма
def optimize_dataframe(df):
    dataset_cols_float = df.select_dtypes(include=['float64', 'float32']).columns.tolist()
    dataset_cols_int = df.select_dtypes(include=['int64', 'int32']).columns.tolist()

    for col in dataset_cols_float:
        df[col] = pd.to_numeric(df[col], downcast='float')

    for col in dataset_cols_int:
        df[col] = pd.to_numeric(df[col], downcast='integer')

    return df


import pandas as pd

def get_unique_counts_and_total_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Возвращает DataFrame с двумя колонками:
      - 'feature' — название признака,
      - 'n_unique' — число уникальных значений.
    
    В конце добавляется отдельная строка:
      - feature = 'Общее число записей',
      - n_unique = общее число строк в датафрейме.

    Параметры
    ---------
    df : pd.DataFrame
        Входной датафрейм.

    Возвращает
    ----------
    pd.DataFrame
        Таблица с названиями признаков и числом уникальных значений,
        плюс итоговая строка с общим числом записей.
    """
    # Считаем уникальные значения по колонкам
    unique_counts = df.nunique()
    
    # Создаём DataFrame из результатов
    result = pd.DataFrame({
        'feature': unique_counts.index,
        'n_unique': unique_counts.values
    })
    
    # Добавляем строку с общим числом записей
    total_rows = len(df)
    total_row = pd.DataFrame([{'feature': 'Общее число записей', 'n_unique': total_rows}])
    
    return pd.concat([result, total_row], ignore_index=True)


In [3]:
# Загрузка первичных данных

df_users = pd.read_csv('https://code.s3.yandex.net/datasets/ds_s13_users.csv')
df_visits = pd.read_csv('https://code.s3.yandex.net/datasets/ds_s13_visits.csv')
df_ads_activity = pd.read_csv('https://code.s3.yandex.net/datasets/ads_activity.csv')
df_surf_depth = pd.read_csv('https://code.s3.yandex.net/datasets/surf_depth.csv')
df_primary_device = pd.read_csv('https://code.s3.yandex.net/datasets/primary_device.csv')
df_cloud_usage = pd.read_csv('https://code.s3.yandex.net/datasets/cloud_usage.csv')

### 1.1 Датасет ds_users

In [4]:
df_users = optimize_dataframe(df_users)
df_users.head()

,user_id,age_category
0,f545-8c95aefe8d3e5548a689-a5b2fd39,4
1,cb48-5a0d6cde4d86ae10637e-c8ceb6ed,2
2,678b-614cd47d854b9d591db2-000b2e50,0
3,4ac0-dad169100b4a29b20818-b26ae7c5,4
4,f19b-9ac21ca973b41ecfa8c3-6a58191d,0


In [5]:
df_users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5913 entries, 0 to 5912
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   user_id       5913 non-null   object
 1   age_category  5913 non-null   int8  
dtypes: int8(1), object(1)
memory usage: 52.1+ KB


In [6]:
# В наличии дубли
get_unique_counts_and_total_df(df_users)

,feature,n_unique
0,user_id,5826
1,age_category,5
2,Общее число записей,5913


In [ ]:
# Явные дубли
df_users.duplicated().sum()

np.int64(87)

In [ ]:
# Явные дубли устранены
# Первичный ключ user_id
df_users = df_users.drop_duplicates()
get_unique_counts_and_total_df(df_users)

,feature,n_unique
0,user_id,5826
1,age_category,5
2,Общее число записей,5826


### 1.2 Датасет df_visits

In [7]:
df_visits = optimize_dataframe(df_visits)
df_visits.head()

,date,daytime,session_id,user_id,website_category
0,2025-11-01,вечер,066e4e02-8c1f-45eb-a50f-178659abe698,0010-5cf8f6b38a7b6c70a021-009dbcda,Category 17
1,2025-11-01,вечер,0bce1749-3376-439c-9a22-f8ffbba00e9a,0010-5cf8f6b38a7b6c70a021-009dbcda,Category 19
2,2025-11-01,вечер,3445d8c4-221d-4d88-bb6a-a2939fe3c610,0010-5cf8f6b38a7b6c70a021-009dbcda,Category 18
3,2025-11-01,вечер,3bf97286-1d91-4aaa-af4a-ed58eceb8cd2,0010-5cf8f6b38a7b6c70a021-009dbcda,Category 20
4,2025-11-01,вечер,40e22712-3cad-410d-a9f0-13bd8f6911c0,0010-5cf8f6b38a7b6c70a021-009dbcda,Category 05


In [8]:
df_visits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1065745 entries, 0 to 1065744
Data columns (total 5 columns):
 #   Column            Non-Null Count    Dtype 
---  ------            --------------    ----- 
 0   date              1065745 non-null  object
 1   daytime           1065745 non-null  object
 2   session_id        1065745 non-null  object
 3   user_id           1065745 non-null  object
 4   website_category  1065745 non-null  object
dtypes: object(5)
memory usage: 40.7+ MB


In [ ]:
# Явные дубли в наличии
get_unique_counts_and_total_df(df_visits)

,feature,n_unique
0,date,14
1,daytime,4
2,session_id,1049995
3,user_id,5826
4,website_category,20
5,Общее число записей,1065745


In [26]:
df_visits.duplicated().sum()

np.int64(15750)

In [ ]:
# Явные дубли устранены
# Первичный ключ session_id
df_visits = df_visits.drop_duplicates()
get_unique_counts_and_total_df(df_visits)

,feature,n_unique
0,date,14
1,daytime,4
2,session_id,1049995
3,user_id,5826
4,website_category,20
5,Общее число записей,1049995


### 1.3 Датасет df_ads_activity

In [10]:
df_ads_activity = optimize_dataframe(df_ads_activity)
df_ads_activity.head()

,user_id,ads_activity
0,e318-d8e69c86b543a5fb927c-c36fb6e6,очень часто
1,35cd-a972339dec534f49332c-a8b6d383,редко
2,f7e6-3b29cf9cb7ed4bb00d8f-81534360,очень редко
3,5186-e25a37549e50f45b2b43-178eaabe,умеренно
4,febd-077f277466253ee04ef6-42656680,умеренно


In [11]:
df_ads_activity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5826 entries, 0 to 5825
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   user_id       5826 non-null   object
 1   ads_activity  5826 non-null   object
dtypes: object(2)
memory usage: 91.2+ KB


In [12]:
# В наличии дубли
get_unique_counts_and_total_df(df_ads_activity)

,feature,n_unique
0,user_id,5593
1,ads_activity,5
2,Общее число записей,5826


In [ ]:
# Явные дубли
df_ads_activity.duplicated().sum()

np.int64(233)

In [ ]:
# Явные дубли устранены
# Первичный ключ user_id
df_ads_activity = df_ads_activity.drop_duplicates()
get_unique_counts_and_total_df(df_ads_activity)

,feature,n_unique
0,user_id,5593
1,ads_activity,5
2,Общее число записей,5593


### 1.4 Датасет df_surf_depth

In [13]:
df_surf_depth = optimize_dataframe(df_surf_depth)
df_surf_depth.head()

,user_id,surf_depth
0,f238-0c4c1e787cce311541b7-736925a0,поверхностно
1,9030-1b562ad80182b6dc27f1-ce811740,глубоко
2,22e0-7c6cadcc45e246b8688d-c43c9b23,поверхностно
3,9d7f-a19f10756378940a49b5-5d03e1ef,поверхностно
4,4233-bb5ae4b09827e5497094-1a4956af,глубоко


In [14]:
df_surf_depth.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5715 entries, 0 to 5714
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     5715 non-null   object
 1   surf_depth  5715 non-null   object
dtypes: object(2)
memory usage: 89.4+ KB


In [ ]:
# Первичный ключ user_id
get_unique_counts_and_total_df(df_surf_depth)

,feature,n_unique
0,user_id,5715
1,surf_depth,3
2,Общее число записей,5715


### 1.5 Датасет df_primary_device

In [16]:
df_primary_device = optimize_dataframe(df_primary_device)
df_primary_device.head()

,user_id,primary_device
0,d602-ec060db7597a6b8cd4e7-aa625896,смартфон
1,9204-9558455be649d4e77945-b5e25d62,ПК
2,5eea-22babd6a9474b43b9d0b-a39a4cf2,ноутбук
3,c142-0296948e8d08e417de10-2da9523c,смартфон
4,abec-bb4092da51eb2233a928-e44ba074,ПК


In [17]:
df_primary_device.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5669 entries, 0 to 5668
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   user_id         5669 non-null   object
 1   primary_device  5669 non-null   object
dtypes: object(2)
memory usage: 88.7+ KB


In [ ]:
# Первичный ключ user_id
get_unique_counts_and_total_df(df_primary_device)

,feature,n_unique
0,user_id,5669
1,primary_device,4
2,Общее число записей,5669


### 1.6 Датасет df_cloud_usage

In [ ]:
# Первичный ключ user_id
df_cloud_usage = optimize_dataframe(df_cloud_usage)
df_cloud_usage.head()

,user_id,cloud_usage
0,a1e4-91c8a52eb855595e653f-298ce305,False
1,db9a-7b8e9e94448b7fcb19b6-4edca15f,False
2,0d55-9ad768879e9b08ca7ff9-843f76c7,True
3,4baa-43285d10a6d3cc969f2a-b21881d1,False
4,b8cd-cbb2411db005115ca64d-32700c62,False


In [20]:
df_cloud_usage.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5680 entries, 0 to 5679
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   user_id      5680 non-null   object
 1   cloud_usage  5680 non-null   bool  
dtypes: bool(1), object(1)
memory usage: 50.1+ KB


In [21]:
get_unique_counts_and_total_df(df_cloud_usage)

,feature,n_unique
0,user_id,5680
1,cloud_usage,2
2,Общее число записей,5680


## Комментарии по итогам подготовки среды и библиотек

Загружены основные библиотеки, подготовлены функции для использования в ходе всего проекта, заданы константы и сделаны необходимые настройки предупреждений.  

Успешно загружены 6 датасетов: `df_users`, `df_visits`,`df_ads_activity`, `df_surf_depth`, `df_primary_device`, `df_cloud_usage`.

1) Датасет `df_users` - содержит данные со значениями целевой переменной **age_category**.  
Состав: 5915 строк с дублями. После устранения дублей получаем первичный ключ **user_id** - 5826 уникальных пользователей.  

2) `df_visits` - лог посещений пользователями ваб-сайтов. **1 065 745** строк, **15 750** явных дублей. Дубли удалены. Первичный уникальный ключ - **session_id** (1 049 995 уникальных записи). Также датасет содержит признаки date (14 уникальных записей), daytime (4 уникальных записи), **user_id** - внешний ключ 5826 записей, website_category (20 уникальных значений).

3) `df_ads_activity`. 5826 записей. Содержал явные дубли. После устранения дублей получен первичный ключ **user_id** (5593 значения) и признак ads_activity с 5 уникальными значениями.

4) Датасет `df_surf_depth`. Не содержит дублей. 5715 записей. Первичный ключ **user_id**, признак surf_depth - 3 уникальных значения.

5) `df_primary_device`: 5569 строк без явных дублей. Первичный ключ **user_id**, признак primary_device имеет 4 уникальных значения.

6) Источник `df_cloud_usage` состоит из 5580 строк. Первичный ключ **user_id**, бинарный признак cloud_usage имеет 2 значения True/False.

Итого: объединение всех 6-ти датасетов будем проводить по ключу **user_id**. Число уникальных записей по первичному ключу совпадает не во всех датасетах, значит будут пропуски при соединении по left join. Нужно определиться с методами их заполнения. Данные логов `df_visits` необходимо сгруппировать и привести к виду, где **user_id** также будет первичным ключом.

____

## Исследовательский анализ данных

In [ ]:
import sweetviz as sv
report = sv.analyze(df_visits)
report.show_html("eda_report.html")

## Предобработка данных

In [ ]:
# Загружаем базовые библиотеки
import numpy as np
import pandas as pd

# Загружаем библиотеки для визуализации данных
import matplotlib.pyplot as plt
import seaborn as sns

# Проверка наличия jinja2 для отражения тепловой карты корреляций
try:
    import jinja2
except:
    %pip install jinja2

# Загружаем библиотеку для расчёта коэффициента корреляции Phik
try:
    from phik import phik_matrix
except:
    %pip install phik
    from phik import phik_matrix

# Предобрабока данных
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import mutual_info_classif, chi2, SelectKBest, VarianceThreshold, SequentialFeatureSelector, RFE

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from category_encoders import LeaveOneOutEncoder
from sklearn.model_selection import cross_validate, StratifiedKFold

# Загружаем модели
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Модули для построения диаграммы калибровки
from sklearn.calibration import calibration_curve, CalibrationDisplay, CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

# Метрики
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, precision_recall_curve, brier_score_loss, log_loss

# Системные библиотеки, настройки отображения датафреймов и предупредительных сообщений
import joblib
import os
import warnings

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning) 

path = os.getcwd()
RANDOM_SEED = 555

## Обучение и оценка базовой модели

## Создание и отбор признаков

## Подбор гиперпараметров моделей

## Подготовка артефактов модели для внедрения

## Выводы о результатах работы